## Imports

In [1]:
import random
import torch
from torch import nn
from torch.amp import GradScaler
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from sklearn import preprocessing
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from matplotlib import colormaps
import pandas as pd
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

seed = 7
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

scaler = GradScaler(device=device)

# because we do NOT want to see 50 plots in our notebook
plt.ioff()

# for mapping labels
label_encoder = preprocessing.LabelEncoder()

# toggle this to retrain the probes
trainToggle = True

Using device: cuda


## constants

In [2]:
# CONSTANTS --------------------------------------------------------------------------

phone_map = {'aa': 0, 'ae': 1, 'ah': 2, 'ao': 3, 'aw': 4, 'ax': 5, 'ax-h': 6, 'axr': 7, 
'ay': 8, 'b': 9, 'bcl': 10, 'ch': 11, 'd': 12, 'dcl': 13, 'dh': 14, 'dx': 15, 
'eh': 16, 'el': 17, 'em': 18, 'en': 19, 'eng': 20, 'epi': 21, 'er': 22, 'ey': 23, 
'f': 24, 'g': 25, 'gcl': 26, 'h#': 27, 'hh': 28, 'hv': 29, 'ih': 30, 'ix': 31, 
'iy': 32, 'jh': 33, 'k': 34, 'kcl': 35, 'l': 36, 'm': 37, 'n': 38, 'ng': 39, 
'nx': 40, 'ow': 41, 'oy': 42, 'p': 43, 'pau': 44, 'pcl': 45, 'q': 46, 'r': 47, 
's': 48, 'sh': 49, 't': 50, 'tcl': 51, 'th': 52, 'uh': 53, 'uw': 54, 'ux': 55, 
'v': 56, 'w': 57, 'y': 58, 'z': 59, 'zh': 60}

# from TIMIT PHONCODE.DOC
moa_map = {
    'stop': {'b', 'd', 'g', 'p', 't', 'k', 'dx', 'q'},
    'affricate': {'jh', 'ch'},
    'fricative': {'s', 'sh', 'z', 'zh', 'f', 'th', 'v', 'dh'},
    'nasal': {'m', 'n', 'ng', 'em', 'en', 'eng', 'nx'},
    'approximant': {'l', 'r', 'w', 'y', 'hh', 'hv', 'el'},
    'vowel': {'iy', 'ih', 'eh', 'ey', 'ae', 'aa', 'aw', 'ay', 'ah', 'ao', 'oy', 'ow', 'uh', 'uw', 'ux', 'er', 'ax', 'ix', 'axr', 'ax-h'},
    'none': {'h#', 'pau', 'epi', '1', '2'}
}

voiced_map = {
    'voiced': {'b', 'd', 'g', 'dx', 'jh', 'z', 'zh', 'v', 'dh', 'm', 'n', 'ng', 'em', 'en', 'eng', 'nx', 'l', 'r', 'w', 'y', 'el', 'iy', 'ih', 'eh', 'ey', 'ae', 'aa', 'aw', 'ay', 'ah', 'ao', 'oy', 'ow', 'uh', 'uw', 'ux', 'er', 'ax', 'ix', 'axr', 'ax-h'},
    'unvoiced': {'p', 't', 'k', 'q', 'ch', 's', 'sh', 'f', 'th', 'hh', 'hv'}
}

# vowel_height_map = {
#     'high': {'iy', 'ih', 'ey', 'uh','uw','ux','ix'},
#     'mid': {'eh', 'ae', 'ah', 'ao', 'er', 'ax', 'axr', 'ax-h'},
#     'low': {'aa', },
#     'diphthong': {'aw', 'ay', 'oy', 'ow'}
# }

# invert the map so we can convert from phones to manner of articulation
phone_to_moa_map = {}
for moa, phone_set in moa_map.items():
    for phone in phone_set:
        phone_to_moa_map[phone] = moa

def phone_to_moa(phone):
    if phone in phone_to_moa_map.keys():
        return phone_to_moa_map[phone]
    return 'none'

# invert the map so we can convert from phones to voicedness
phone_to_voiced_map = {}
for voicedness, phone_set in moa_map.items():
    for phone in phone_set:
        phone_to_voiced_map[phone] = voicedness

def phone_to_voiced(phone):
    if phone in phone_to_voiced_map.keys():
        return phone_to_voiced_map[phone]
    return 'none'

### dataset and probe definitions

In [3]:
# dataset definition
class AudioDataset(Dataset):
    def __init__(self, x, y):
        self.x         = x
        self.y         = y

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.x)

# define probe architecture
class ProbeNet(nn.Module):
    def __init__(self, embedding_dim, encoder="", encoderLayer=-1):
        super(ProbeNet, self).__init__()

        self.encoder = encoder
        self.encoderLayer = encoderLayer

        self.layers = nn.Sequential(
            nn.Linear(embedding_dim, 200),
            nn.ReLU(),
            # nn.Dropout(0.2),
            # nn.Linear(200, 61) # 61 phones
            nn.Linear(200, 6) # 6 manners of articulation
        )
    
    def forward(self, x):
        x = self.layers(x)
        return x

## helper functions

In [ ]:
def data_setup(embeddings, labels, validation_frac=0.2):
    # go from raw phones to MOA labels
    labels = list(map(phone_to_moa, labels))

    # filter out none values
    new_embeddings = []
    new_labels = []

    for i in range(len(labels)):
        if labels[i] != 'none':
            new_embeddings.append(embeddings[i])
            new_labels.append(labels[i])

    # convert our phone labels into integers so we can fit to them
    labels_types = label_encoder.fit_transform(new_labels)

    print("records: " + str(len(new_embeddings)))

    dataset = AudioDataset(new_embeddings, labels_types)
    validation_size = int(len(dataset) * validation_frac)
    train_size = len(dataset) - validation_size

    train_set, validation_set = torch.utils.data.random_split(
        dataset,
        [train_size, validation_size],
        generator=torch.Generator().manual_seed(seed)
    )

    train_loader = DataLoader(
        dataset=train_set,
        batch_size=128,
        shuffle=True,
        num_workers=0
    )

    validation_loader = DataLoader(
        dataset=validation_set,
        batch_size=128,
        shuffle=False,
        num_workers=0
    )

    print('train batches: ' + str(len(train_loader)))
    print('validate batches: ' + str(len(validation_loader)))

    return train_loader, validation_loader

def train(probe, train_loader, validation_loader, writer, epochs, criterion, optimizer, note):

    best_validation_loss = float("inf")
    best_model = None
    epochs_without_improvement = 0

    for epoch in tqdm(range(1, epochs + 1),
            "Training " + probe.encoder + " layer " + str(probe.encoderLayer),
            epochs
    ):
        # trainnig ---------------------------------------------------
        probe.train()

        correct = 0
        total = 0
        training_loss = 0.0

        for data in train_loader:
            inputs = data[0][probe.encoderLayer].to(device).float()
            targets = data[1].to(device)

            # for slice in data:
            #     print(slice[probe.encoderLayer].shape)
            # print(data[probe.encoderLayer].shape)

            outputs = probe(inputs)

            loss = criterion(outputs, targets)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            training_loss += loss.item()
            predicted = torch.argmax(outputs, dim=1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
        
        train_accuracy = 100.*correct / total
        writer.add_scalar(note + "/train_accuracy", train_accuracy, epoch)

        # validation ------------------------------------------------
        probe.eval()
        validation_loss = 0.0
        validation_correct = 0
        validation_total = 0

        with torch.no_grad():
            for data in validation_loader:
                inputs = data[0][probe.encoderLayer].to(device).float()
                targets = data[1].to(device)
                outputs = probe(inputs)
  
                loss = criterion(outputs, targets)
    
                validation_loss += loss.item()
                predicted = torch.argmax(outputs, dim=1)
                validation_total += targets.size(0)
                validation_correct += predicted.eq(targets).sum().item()

        validation_accuracy = validation_correct / validation_total
        writer.add_scalar(note + "/validation_accuracy", validation_accuracy, epoch)

        if validation_loss < best_validation_loss - 0.001:
            best_validation_loss = validation_loss
            best_model = copy.deepcopy(probe.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= 5:
            print("Early stopping...")
            break

    # probe is passed by reference, if it was improving for all epochs then we do not need to reload a previous state
    if best_model is not None:
        probe.load_state_dict(best_model)


def test(probe, loader, writer):
    labels_for_confusion = []
    predictions_for_confusion = []

    total = 0
    correct = 0

    for data in tqdm(
        loader,
        "Testing " + probe.encoder + " layer " + str(probe.encoderLayer),
        len(loader)
    ):
        inputs = data[0][probe.encoderLayer].to(device).float()
        targets = data[1].to(device)

        outputs = probe(inputs)

        predicted = torch.argmax(outputs, dim=1)
        total += predicted.size(0)
        correct += predicted.eq(targets).sum().item()

        cpu_target = targets.cpu()
        cpu_predicted = predicted.cpu()
        
        moa_targets = label_encoder.inverse_transform(cpu_target)
        moa_predicted = label_encoder.inverse_transform(cpu_predicted)

        labels_for_confusion.extend(moa_targets)
        predictions_for_confusion.extend(moa_predicted)
        
    accuracy = 100.*correct/total
    writer.add_scalar(probe.encoder + str(probe.encoderLayer) + ' test accuracy', accuracy)

    results_df = pd.DataFrame({
        "true_label": labels_for_confusion,
        "predicted_label": predictions_for_confusion
        })

    print("writing to " + "predictions/" + probe.encoder + "MOA/layer" + str(probe.encoderLayer) + ".csv")

    results_df.to_csv(
        "predictions/" + probe.encoder + "MOA/layer" + str(probe.encoderLayer) + ".csv",
        index=False
    )

def createConfusionFigure(encoder, encoderLayer):
    results_df = pd.read_csv("predictions/" + encoder + "MOA/layer" + str(encoderLayer) + ".csv")

    table_data = []

    for class_name in label_encoder.classes_:
        class_rows = results_df["true_label"] == class_name
        correct = results_df.loc[class_rows, "predicted_label"] == class_name

        class_accuracy = 100 * correct.sum() / class_rows.sum()
        table_data.append([class_name, f"{class_accuracy:.4f}%"])

    labels_for_confusion = results_df["true_label"].tolist()
    predictions_for_confusion = results_df["predicted_label"].tolist()

    fig, ax = plt.subplots(figsize=(10, 7))
    cm = confusion_matrix(labels_for_confusion, predictions_for_confusion, labels=label_encoder.classes_, normalize="true")

    ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_).plot(ax=ax)
    plt.title(encoder + " layer " + str(encoderLayer) + " Manner of Articulation")

    ax_table = fig.add_axes([0.85, 0.4, 0.20, 0.4])  # [left, bottom, width, height]
    ax_table.axis('off')
    table = ax_table.table(
        cellText=table_data,
        colLabels=['Class', 'Accuracy'],
        loc='center',
        cellLoc='center'
    )
    table.scale(1, 2)

    plt.savefig('figures/' + encoder + "/layer" + str(encoderLayer) + 'MOAconfusion.png', dpi=300, bbox_inches='tight')
    plt.close(fig)

## Whisper
----

### train whisper probes

In [5]:
if trainToggle:
    # load whisper train embeddings
    print("Loading saved whisper embeddings...")
    whisper_train_embeddings = torch.load('data/saved_embeddings/whisper_train_embeddings.pt')
    whisper_train_labels = torch.load('data/saved_embeddings/whisper_train_labels.pt')


Loading saved whisper embeddings...


In [6]:
if trainToggle:
    # set up whisper loader and label encoder
    print("Remapping + batching saved whisper embeddings....")
    whisper_train_loader, whisper_validate_loader = data_setup(whisper_train_embeddings, whisper_train_labels)
    torch.save(label_encoder, 'label_encoders/moa_label_encoder.pt') # data-setup has set up the label encoder that we will reuse

    for layerNumber in range(12):
        # train + save whisper probe per layer of embedding
        whisper_probe = ProbeNet(768, "whisper", layerNumber).to(device)
        optimizer = torch.optim.Adam(whisper_probe.parameters())
        criterion = nn.CrossEntropyLoss()

        writer = SummaryWriter(comment='whisper moa probe, layer ' + str(layerNumber))
        train(
            probe=whisper_probe,
            train_loader=whisper_train_loader,
            validation_loader=whisper_validate_loader,
            writer=writer,
            epochs=50,
            criterion=criterion,
            optimizer=optimizer,
            note='whisper moa probe accuracy, layer ' + str(layerNumber)
        )

        torch.save(whisper_probe, 'models/' + whisper_probe.encoder + 'MOA/layer' + str(whisper_probe.encoderLayer) + '.pth')

    # clear data that we're done with
    del whisper_train_embeddings
    del whisper_train_loader
    del whisper_train_labels

Remapping + batching saved whisper embeddings....
records: 128721
train batches: 805
validate batches: 202


Training whisper layer 0:  20%|██        | 10/50 [00:29<01:58,  2.95s/it]


Early stopping...


Training whisper layer 1:  68%|██████▊   | 34/50 [01:30<00:42,  2.65s/it]


Early stopping...


...

Early stopping...


Training whisper layer 10:  24%|██▍       | 12/50 [00:33<01:45,  2.77s/it]


Early stopping...


Training whisper layer 11:  28%|██▊       | 14/50 [00:38<01:38,  2.75s/it]

Early stopping...


### test probe + display confusion matrix

In [7]:
# whisper MOA test
print("Loading saved test whisper embeddings...")
whisper_test_embeddings = torch.load('data/saved_embeddings/whisper_test_embeddings.pt')
whisper_test_labels = torch.load('data/saved_embeddings/whisper_test_labels.pt')


Loading saved test whisper embeddings...


In [8]:
print("Remapping + batching saved test whisper embeddings....")
whisper_test_loader = data_setup(whisper_test_embeddings, whisper_test_labels, validation_frac=0.0)[0]

writer = SummaryWriter(comment='whisper moa probe')
label_encoder = torch.load('label_encoders/moa_label_encoder.pt', weights_only=False)

for layerNumber in range(12):
    encoder = 'whisper'
    whisper_probe = torch.load('models/' + encoder + 'MOA/layer' + str(layerNumber) + '.pth', weights_only=False)

    whisper_probe.eval()
    test(whisper_probe, whisper_test_loader, writer)

    createConfusionFigure("whisper", layerNumber)

del whisper_test_embeddings
del whisper_test_labels

Remapping + batching saved test whisper embeddings....
records: 47013
train batches: 368
validate batches: 0


Testingwhisper layer 0: 100%|██████████| 368/368 [00:00<00:00, 373.95it/s]


writing to predictions/whisperMOA/layer0.csv


Testingwhisper layer 1: 100%|██████████| 368/368 [00:00<00:00, 402.78it/s]


writing to predictions/whisperMOA/layer1.csv


...

writing to predictions/whisperMOA/layer9.csv


Testingwhisper layer 10: 100%|██████████| 368/368 [00:00<00:00, 439.88it/s]


writing to predictions/whisperMOA/layer10.csv


Testingwhisper layer 11: 100%|██████████| 368/368 [00:00<00:00, 438.93it/s]


writing to predictions/whisperMOA/layer11.csv


## wav2vec
---
### train wav2vec

In [9]:
# initialize and train the wav2vec probe
if trainToggle:
    print("Loading saved wav2vec embeddings...")
    wav2vec_train_embeddings = torch.load('data/saved_embeddings/wav2vec_train_embeddings.pt')
    wav2vec_train_labels = torch.load('data/saved_embeddings/wav2vec_train_labels.pt')


Loading saved wav2vec embeddings...


In [10]:
if trainToggle:
    # set up train loader for wav2vec
    print("Remapping + batching saved wav2vec embeddings....")
    wav2vec_train_loader, wav2vec_validate_loader = data_setup(wav2vec_train_embeddings, wav2vec_train_labels)

    for layerNumber in range(12):
        # train + save probe per layer of embedding
        wav2vec_probe = ProbeNet(768, "wav2vec", layerNumber).to(device)
        optimizer = torch.optim.Adam(wav2vec_probe.parameters())
        criterion = nn.CrossEntropyLoss()

        writer = SummaryWriter(comment='wav2vec probe, layer ' + str(layerNumber))
        train(
            probe=wav2vec_probe,
            train_loader=wav2vec_train_loader,
            validation_loader=wav2vec_validate_loader,
            writer=writer,
            epochs=50,
            criterion=criterion,
            optimizer=optimizer,
            note='wav2vec moa probe accuracy, layer ' + str(layerNumber)
        )

        torch.save(wav2vec_probe, 'models/' + wav2vec_probe.encoder + 'MOA/layer' + str(wav2vec_probe.encoderLayer) + '.pth')


    # clear data that we're done with
    del wav2vec_train_embeddings
    del wav2vec_train_loader
    del wav2vec_train_labels

Remapping + batching saved wav2vec embeddings....
records: 128721
train batches: 805
validate batches: 202


Training wav2vec layer 0:  22%|██▏       | 11/50 [00:32<01:53,  2.92s/it]


Early stopping...


Training wav2vec layer 1:  20%|██        | 10/50 [00:28<01:53,  2.84s/it]


Early stopping...


...

Early stopping...


Training wav2vec layer 10:  32%|███▏      | 16/50 [00:51<01:49,  3.22s/it]


Early stopping...


Training wav2vec layer 11:  48%|████▊     | 24/50 [01:11<01:17,  3.00s/it]

Early stopping...


### test wav2vec + display confusion matrix

In [11]:
# test wav2vec MOA probe
print("Loading saved test wav2vec embeddings...")
wav2vec_test_embeddings = torch.load('data/saved_embeddings/wav2vec_test_embeddings.pt')
wav2vec_test_labels = torch.load('data/saved_embeddings/wav2vec_test_labels.pt')


Loading saved test wav2vec embeddings...


In [12]:
print("Remapping + batching saved test wav2vec embeddings....")
wav2vec_test_loader = data_setup(wav2vec_test_embeddings, wav2vec_test_labels, validation_frac=0.0)[0]

writer = SummaryWriter(comment='wav2vec moa probe')
label_encoder = torch.load('label_encoders/moa_label_encoder.pt', weights_only=False)

for layerNumber in range(12):
    encoder = 'wav2vec'
    wav2vec_probe = torch.load('models/' + encoder + 'MOA/layer' + str(layerNumber) + '.pth', weights_only=False)

    wav2vec_probe.eval()
    test(wav2vec_probe, wav2vec_test_loader, writer)

    createConfusionFigure("wav2vec", layerNumber)

del wav2vec_test_embeddings
del wav2vec_test_labels

Remapping + batching saved test wav2vec embeddings....
records: 47013
train batches: 368
validate batches: 0


Testingwav2vec layer 0: 100%|██████████| 368/368 [00:01<00:00, 360.59it/s]


writing to predictions/wav2vecMOA/layer0.csv


Testingwav2vec layer 1: 100%|██████████| 368/368 [00:00<00:00, 386.60it/s]


writing to predictions/wav2vecMOA/layer1.csv


...

writing to predictions/wav2vecMOA/layer9.csv


Testingwav2vec layer 10: 100%|██████████| 368/368 [00:00<00:00, 400.34it/s]


writing to predictions/wav2vecMOA/layer10.csv


Testingwav2vec layer 11: 100%|██████████| 368/368 [00:00<00:00, 405.86it/s]


writing to predictions/wav2vecMOA/layer11.csv


## voxtral
---
### train

In [13]:
if trainToggle:
    # initialize and train the voxtral probe
    print("Loading saved voxtral embeddings...")
    voxtral_train_embeddings = torch.load('data/saved_embeddings/voxtral_train_embeddings.pt')
    voxtral_train_labels = torch.load('data/saved_embeddings/voxtral_train_labels.pt')


Loading saved voxtral embeddings...


In [14]:
if trainToggle:
    # set up voxtral loader
    print("Remapping + batching saved voxtral embeddings....")
    voxtral_train_loader, voxtral_validate_loader = data_setup(voxtral_train_embeddings, voxtral_train_labels)

    for layerNumber in range(32):
        # train + save probe per layer
        voxtral_probe = ProbeNet(1280, "voxtral", layerNumber).to(device)
        optimizer = torch.optim.Adam(voxtral_probe.parameters())
        criterion = nn.CrossEntropyLoss()

        writer = SummaryWriter(comment='voxtral probe, layer ' + str(layerNumber))
        train(
            probe=voxtral_probe,
            train_loader=voxtral_train_loader,
            validation_loader=voxtral_validate_loader,
            writer=writer,
            epochs=50,
            criterion=criterion,
            optimizer=optimizer,
            note='voxtral moa probe accuracy, layer ' + str(layerNumber)
        )

        torch.save(voxtral_probe, 'models/' + voxtral_probe.encoder + 'MOA/layer' + str(voxtral_probe.encoderLayer) + '.pth')

    # clear data that we're done with
    del voxtral_train_embeddings
    del voxtral_train_loader
    del voxtral_train_labels

Remapping + batching saved voxtral embeddings....
records: 128721
train batches: 805
validate batches: 202


Training voxtral layer 0:  86%|████████▌ | 43/50 [03:50<00:37,  5.35s/it]


Early stopping...


Training voxtral layer 1:  84%|████████▍ | 42/50 [03:45<00:43,  5.38s/it]


Early stopping...


...

Early stopping...


Training voxtral layer 30:  28%|██▊       | 14/50 [01:16<03:17,  5.49s/it]


Early stopping...


Training voxtral layer 31:  20%|██        | 10/50 [00:59<03:57,  5.95s/it]

Early stopping...


### test + confusion matrix

In [15]:
# voxtral MOA test
print("Loading saved test voxtral embeddings...")
voxtral_test_embeddings = torch.load('data/saved_embeddings/voxtral_test_embeddings.pt')
voxtral_test_labels = torch.load('data/saved_embeddings/voxtral_test_labels.pt')


Loading saved test voxtral embeddings...


In [16]:

print("Remapping + batching saved test voxtral embeddings....")
voxtral_test_loader = data_setup(voxtral_test_embeddings, voxtral_test_labels, validation_frac=0.0)[0]

writer = SummaryWriter(comment='voxtral moa probe')
label_encoder = torch.load('label_encoders/moa_label_encoder.pt', weights_only=False)

for layerNumber in range(32):
    encoder = 'voxtral'
    voxtral_probe = torch.load('models/' + encoder + 'MOA/layer' + str(layerNumber) + '.pth', weights_only=False)

    voxtral_probe.eval()

    test(voxtral_probe, voxtral_test_loader, writer)

    createConfusionFigure("voxtral", layerNumber)

Remapping + batching saved test voxtral embeddings....
records: 47013
train batches: 368
validate batches: 0


Testingvoxtral layer 0: 100%|██████████| 368/368 [00:01<00:00, 206.64it/s]


writing to predictions/voxtralMOA/layer0.csv


Testingvoxtral layer 1: 100%|██████████| 368/368 [00:01<00:00, 213.18it/s]


writing to predictions/voxtralMOA/layer1.csv


...

writing to predictions/voxtralMOA/layer29.csv


Testingvoxtral layer 30: 100%|██████████| 368/368 [00:01<00:00, 216.29it/s]


writing to predictions/voxtralMOA/layer30.csv


Testingvoxtral layer 31: 100%|██████████| 368/368 [00:01<00:00, 207.88it/s]


writing to predictions/voxtralMOA/layer31.csv
